## Orchestrator-Workers Workflow
In this workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

### When to use this workflow
This workflow is well-suited for complex tasks where you can't predict the subtasks needed. The key difference from simple parallelization is its flexibility—subtasks aren't pre-defined, but determined by the orchestrator based on the specific input.

### Improvements from [original cookbook notebook](https://github.com/anthropics/anthropic-cookbook/blob/main/patterns/agents/orchestrator_workers.ipynb)

- The `context` variable for the task is unused so its been removed (delete unused code).
- Original prompt tended to lead the model too much (prompt engineering).
- Original doesn't actually synthesize the results after using the workers, added it in (meet spec).

### Usage Notes
- Needs better error handling when structure isn't met. claude-3-5-haiku couldn't keep to XML spec at all.


In [1]:

# load required gems and add some helpers for pretty printing in iruby
require_relative "../../../../notebook" 

require "rexml/document"
def extract_xml(text, path)
    xml = REXML::Document.new("<root>#{text}</root>") # Wrapping in <root> to make valid XML
    xml.elements["root/"+ path].text.strip
end

# set the default model
Instruct.set_default_model "claude-3-5-sonnet-latest", access_token: ENV['ANTHROPIC_API_KEY']

true

In [2]:
task="Write a product description for a new eco-friendly water bottle"

orchestrator_prompt = p.user{"
Analyze this task and break it down into 2-3 distinct approaches:

Task: <%= task %>

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
        <style>[writing style]</style>
        <guidelines>[writing guidelines]</guidelines>
    </task>
</tasks>
"} + gen

result = orchestrator_prompt.call(temperature: 0.1)
puts result



<analysis>
This task requires creating marketing copy for a water bottle with environmental benefits. Key variations would be valuable based on:
1. Emotional vs. technical focus
2. Length and detail level
3. Primary selling point (sustainability vs. functionality)

Different approaches serve distinct purposes:
- A technical approach builds credibility and appeals to environmentally conscious consumers who want specifics
- An emotional/lifestyle approach connects the product to values and daily habits
- A features-first approach highlights practical benefits while weaving in eco-friendly aspects
</analysis>

<tasks>
    <task>
        <style>Technical and factual with environmental emphasis</style>
        <guidelines>
        - Lead with specific eco-credentials (recycled materials %, carbon footprint)
        - Include technical specifications and materials science
        - Use data points and comparisons
        - Maintain professional, authoritative tone
        - Focus on environm

In [3]:
xml =  REXML::Document.new("<root>#{result}</root>")
puts "Analysis:\n#{xml.elements['root/analysis'].text.strip}"
worker_tasks = []
REXML::XPath.each(xml,'root/tasks/task') do |task|
    worker_tasks << { style: task.elements['style'].text.strip, guidelines: task.elements['guidelines'].text.strip }
end

worker_prompts = worker_tasks.map do |worker_task|
    p.user{"Generate content based on:
Task: <%= task %>
Style: <%= worker_task[:style] %>
Guidelines: <%= worker_task[:guidelines] %>

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"} + gen
end

worker_results = worker_prompts.map(&:call).map{ |result| extract_xml(result, "response") }

Analysis:
This task requires creating marketing copy for a water bottle with environmental benefits. Key variations would be valuable based on:
1. Emotional vs. technical focus
2. Length and detail level
3. Primary selling point (sustainability vs. functionality)

Different approaches serve distinct purposes:
- A technical approach builds credibility and appeals to environmentally conscious consumers who want specifics
- An emotional/lifestyle approach connects the product to values and daily habits
- A features-first approach highlights practical benefits while weaving in eco-friendly aspects


["The EcoVessel Pro Series X1 represents a breakthrough in sustainable hydration technology, manufactured from 94% post-consumer recycled stainless steel with a verified 78% lower carbon footprint compared to conventional water bottles.\n\nTechnical Specifications:\n- Capacity: 750ml (25.4 fl oz)\n- Weight: 295g (10.4 oz)\n- Dimensions: 27.5cm H x 7.6cm D (10.8\" x 3\")\n- Thermal Retention: 24 hours cold / 12 hours hot\n- Impact Resistance: 2.0m drop test certified\n\nMaterial Science Innovation:\nThe X1's proprietary triple-layer construction features a recycled 18/8 stainless steel core, surrounded by a vacuum-sealed thermal barrier and protected by an impact-resistant outer shell derived from reclaimed ocean plastics. Our advanced metallurgical process reduces manufacturing energy consumption by 52% compared to virgin steel production.\n\nEnvironmental Impact Metrics:\n- 2.8kg CO2e lifecycle emissions (63% less than standard bottles)\n- 100% plastic-free packaging\n- Zero microplas

In [10]:
orchestrator_prompt = p.user{"
Take the best of these 2-3 distinct approaches to the same writing task and decide on the best to publish:

Task: <%= task %>
<% worker_results.each do |result| %>
-----
 <%= result %>
<% end %>

Return your response in this format:

<thoughts>
Your thoughts before answer
</thoughts>
<response>
 New version rewritten as well as possible
</response>

"} + gen
result = orchestrator_prompt.call
puts result
puts "\n" + "-" * 40 + "\n"
puts extract_xml(result,"response")

<thoughts>
Analyzing the three versions:
1. First version is very technical and data-heavy - good for B2B or technical buyers but might alienate general consumers
2. Second version is emotionally engaging and lifestyle-focused but could use more specific features
3. Third version has a good balance of practical features and environmental benefits, but could be more compelling

Best approach would be to combine the emotional resonance of #2 with the clear feature presentation of #3, while incorporating select impactful stats from #1. The description should inspire while informing, and make both practical and emotional cases for purchase.
</thoughts>

<response>
Transform Your Daily Hydration with the EcoFlow Thermal Bottle

Make every sip count with the EcoFlow Thermal Bottle, where premium performance meets environmental consciousness. Crafted from recycled, food-grade stainless steel, this thoughtfully engineered bottle isn't just about staying hydrated – it's about making a differenc